# KnowSeek.Ai — EDA Notebook
**Version: rev05_003 — Branch: main_sia05 — 22.03.2026 16:19**
> ⚠️ This notebook requires data files in `05_data/` — see README for setup instructions.

> This notebook is the **window**. The `.py` files are the **engine**.
> All logic lives in `01_backend/modules/`. This notebook calls those files and shows the results.

---

## Chapter Overview

| Chapter | Topic | Status |
|---------|-------|--------|
| 1 | Environment Check | ✅ |
| 2 | Load Data | ✅ |
| 3 | EDA — Explore the Data | ✅ |
| 4 | BM25 Baseline | ✅ |
| 5 | RAG + llama3 | ✅ |
| 6 | Compare BM25 vs RAG | ✅ |

---

**How to use this notebook:**
- Run chapters from top to bottom
- Each chapter can be run independently
- Add `🔴` to a section title if it is not finished in time
- Add `✅` to a section title when it is done

---
# Chapter 1 — Environment Check
> **Goal:** Make sure all tools are running before we start.

We check three things:
- Python environment is active and all packages are installed
- Ollama is running with the right models
- ChromaDB is ready

In [1]:
# ─── Chapter 1 — Imports ──────────────────
import sys
import importlib
import requests
import chromadb
from pathlib import Path
import plotly.io as pio

# ─── Global Paths — used in this chapter ──
BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / "05_data"
DB_PATH   = str(BASE_PATH / "chroma_db")

pio.renderers.default = "notebook"

print("✅ Chapter 1 imports ready")

✅ Chapter 1 imports ready


## 1.1 Virtual Environment Check
**Task:** Check if Python is the right version and all packages are installed.

In [2]:
# 1.1 virtual environment check

print(f"Python version: {sys.version.split()[0]}")
print(f"Expected:       3.11.3")
if sys.version.split()[0] == "3.11.3":
    print("   OK")
else:
    print("   ⚠️  Version mismatch")
print()

packages = [
    "langchain", "chromadb", "rank_bm25", "mlflow",
    "pdfplumber", "fastapi", "pandas", "seaborn",
    "plotly", "nbformat", "pytesseract", "fpdf"
]

missing = []
for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"  OK  {pkg}")
    except ImportError:
        print(f"  ❌  {pkg} — run: pip install {pkg}")
        missing.append(pkg)

print()
if missing:
    print(f"⚠️  Missing packages: {', '.join(missing)}")
else:
    print("✅ All packages ready")

Python version: 3.11.3
Expected:       3.11.3
   OK

  OK  langchain
  OK  chromadb
  OK  rank_bm25
  OK  mlflow
  OK  pdfplumber
  OK  fastapi
  OK  pandas
  OK  seaborn
  OK  plotly
  OK  nbformat
  OK  pytesseract
  OK  fpdf

✅ All packages ready


## 1.2 Ollama Connection Check
**Task:** Check if Ollama is running and the models are available.

In [3]:
# 1.2 Check Ollama

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print("✅ Ollama is running")
    print()

    for required in ["llama3", "nomic-embed-text"]:
        found = any(required in m for m in models)
        status = "✅" if found else "❌"
        print(f"  {status}  {required}")

except Exception as e:
    print(f"❌ Ollama not running: {e}")
    print("   Start with: brew services start ollama")

✅ Ollama is running

  ✅  llama3
  ✅  nomic-embed-text


## 1.3 ChromaDB Setup Check
**Task:** Make sure the ChromaDB folder exists and is ready to store data.

In [4]:
# 1.3 Check ChromaDB

Path(DB_PATH).mkdir(exist_ok=True)
client = chromadb.PersistentClient(path=DB_PATH)
collections = client.list_collections()

print(f"✅ ChromaDB ready")
print(f"   Path: {DB_PATH}")
print()
if collections:
    for c in collections:
        print(f"   Collection: {c.name}")
else:
    print("   ⚠️  No collections yet — run Chapter 2 to ingest data")

✅ ChromaDB ready
   Path: /Users/asimeoa/aipm-1711/KnowSeek/chroma_db

   Collection: docseek


---
# Chapter 2 — Load Data
> **Goal:** Load all files from `05_data/` and prepare them for the AI.

Steps:
1. Find all files
2. Check PDF readability — auto OCR if needed
3. Chunk + ingest per module
4. Check metadata
5. Verify vectorization

In [5]:
# ─── Chapter 2 — Imports ──────────────────
import sys
import pandas as pd
import pdfplumber
from pathlib import Path
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction

# ─── Global Paths — used in this chapter ──
BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / "05_data"
DB_PATH   = str(BASE_PATH / "chroma_db")

# ─── Module Paths ─────────────────────────
sys.path.append(str(BASE_PATH / "01_backend/modules/02_docseek"))
sys.path.append(str(BASE_PATH / "01_backend/modules/01_partseek"))
sys.path.append(str(BASE_PATH / "01_backend/utils"))  # shared tools

import ingest
import check_pdfs  # PDF readability check + auto OCR

print("✅ Chapter 2 imports ready")

✅ Chapter 2 imports ready


## 2.1 File Loader
**Task:** Find all files in `05_data/` and show what we have.

In [6]:
# 2.1 load and check data files

supported = [".pdf", ".png", ".webp", ".jpg", ".docx", ".xlsx"]

files = []
for f in DATA_PATH.rglob("*"):
    if f.is_file() and not f.name.startswith("."):
        files.append({
            "folder": f.parent.name,
            "type":   f.suffix.lower(),
            "size_mb": round(f.stat().st_size / 1024 / 1024, 2)
        })

df = pd.DataFrame(files)

print(f"DATA_PATH:   {DATA_PATH}")
print(f"Total files: {len(df)}")
print()
print(df.groupby("type").agg(count=("type","count"), total_mb=("size_mb","sum")).round(2).to_string())
print()

unsupported = df[~df["type"].isin(supported)]["type"].unique()
if len(unsupported) > 0:
    print(f"⚠️  Cannot vectorize: {', '.join(unsupported)}")
else:
    print("✅ All file types supported")

DATA_PATH:   /Users/asimeoa/aipm-1711/KnowSeek/05_data
Total files: 48

       count  total_mb
type                  
.pdf      39     12.34
.png       8      2.12
.webp      1      0.00

✅ All file types supported


## 2.2 Chunking
**Goal:** Split documents into small pieces so the AI can read them.
Each module handles different file types.

### 2.2.0 PDF Readability Check
**Task:** Check which PDFs have readable text. Auto-convert via OCR if needed.

In [10]:
# 2.2.0 PDF Readability Check
# Calls check_pdfs.py — logic lives there

check_pdfs.run_check(data_path=DATA_PATH)

Checking 39 PDFs in 05_data/

✅ Readable:   39 files — text already extractable
🔄 Converted:  0 files — OCR applied successfully

✅ All files are now readable


{'readable': ['CRASH-ISO-18571 – ISO:TS 18571.pdf',
  'GEORGE-01- Aging & Corrosion Performance Standard_PM.pdf',
  'Großer OEM-Vergleich- Korrosions-Performance (Anonymisiert).pdf',
  '49 CFR Part 572 – Anthropomorphic Test Dummies.pdf',
  'MICKEY-01- Corrosion Performance Master Standard_PG_rev02.pdf',
  'ZEUS-01- Interior & Structure Corrosion Standard_PV.pdf',
  'CRASH-FMVSS-208.pdf',
  'CRASH-EURONCAP-FRONTAL.pdf',
  'MICKEY-01- Corrosion Performance Master Standard_PG.pdf',
  'Delta Done_Doc_PV000_DLT.pdf',
  'Hexagon Nuts with Flannge_PV.pdf',
  'Screw Internal spline drive bolt drive_Doc_PV000_488.pdf',
  'Screw Round head_Doc_PA000_185_(M6-M12).pdf',
  'Screw Serrated flange bolt_Doc_PV000_537.pdf',
  'Flange Screws M-Threads_PV .pdf',
  'DIN EN ISO .pdf',
  'Schrew Bold sw under head_Doc_PA000_185_(M14-M20).pdf',
  'Screw External Torx collar bolt_Doc_PV000_052.pdf',
  'TORX Flange Screw _SIA.pdf',
  'inner Hexagonal Screw SIA.pdf',
  'Screw_Spline drive flange bolt with coll

### 2.2.1 DocSeek — PDF Chunking
**Task:** Load all PDFs from `05_data/` and split into chunks.
**What we show:** How many documents and chunks are indexed and ready for search.
Calls `ingest.py` from the DocSeek backend.

In [8]:
# 2.2.1 DocSeek ingest

summary = ingest.run_ingest(
    data_path=DATA_PATH,
    config_name="medium"
)


─── Ingest Summary ───────────────────
  config_name          medium
  chunk_size           500
  chunk_overlap        100
  total_chunks         121
  total_docs           39
  ingest_time_s        6.47
  collection           docseek
──────────────────────────────────────

✅  All file types supported


### 2.2.2 PartSeek — Part Search
**Task:** Load all fastener datasheets from `05_data/01_Fasteners/` and verify they are ready for part search.
**What we show:** How many parts are indexed with `category=Bolts+Torque` and ready for PartSeek.
Calls `ingest.py` from the PartSeek backend.

In [ ]:
# 2.2.2 PartSeek — verify parts are ready
import ingest as partseek_ingest
partseek_ingest.run_partseek_ingest()

Categories in ChromaDB:
  🔩  Bolts+Torque: 22 chunks
  📄  Corrosion: 43 chunks
  📄  General: 41 chunks
  📄  Painting: 15 chunks

✅ PartSeek ready — 22 Bolts+Torque chunks found


### 2.2.3 NormSeek — Planned
**Status:** ⏳ Phase 2

When active: Add norm PDFs to `05_data/04_Norms/` and run ingest.

### 2.2.4 CostSeek — Planned
**Status:** ⏳ Phase 3

When active: Add cost data to `05_data/05_Cost/` and run ingest.

## 2.3 Tagging & Metadata
**Task:** Verify that every chunk has the correct metadata attached.

In [ ]:
# 2.3 check metadata in ChromaDB

ollama_ef = OllamaEmbeddingFunction(
    url="http://localhost:11434/api/embeddings",
    model_name="nomic-embed-text"
)

client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(
    name="docseek",
    embedding_function=ollama_ef
)

sample = collection.get(limit=3, include=["metadatas"])
print("Sample metadata (3 chunks):")
print()
for m in sample["metadatas"]:
    for k, v in m.items():
        if k != "filename":  # no filenames in output
            print(f"  {k}: {v}")
    print()

## 2.4 Vectorization & Storage
**Task:** Verify that all chunks are stored as vectors in ChromaDB.

In [ ]:
# 2.4 check vector count in ChromaDB

client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(name="docseek")
count = collection.count()

print(f"✅ ChromaDB collection: docseek")
print(f"   Total vectors: {count}")
print()
if count > 0:
    print("✅ Ready for search")
else:
    print("⚠️  No vectors — run ingest first")

---
# Chapter 3 — EDA — Explore the Data
> **Goal:** Understand the data across all modules before building the AI system.

We look at:
- How many files per module?
- What file types do we have?
- How big are the chunks?
- What languages are in the documents?

In [ ]:
# ─── Chapter 3 — Imports ──────────────────
import chromadb
import pdfplumber
import pandas as pd
import plotly.express as px
import plotly.io as pio
from pathlib import Path
from IPython.display import Markdown, display

# ─── Global Paths — used in this chapter ──
BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / "05_data"
DB_PATH   = str(BASE_PATH / "chroma_db")

pio.renderers.default = "notebook"

client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(name="docseek")

print("✅ Chapter 3 imports ready")

## 3.0 Ingest Results
**Task:** Show a summary of what is in ChromaDB right now.

In [ ]:
# 3.0 overview of ingest results

sample = collection.get(limit=collection.count(), include=["metadatas"])
df_meta = pd.DataFrame(sample["metadatas"])

summary = df_meta.groupby("category").size().reset_index(name="chunks")

table = "| Category | Chunks |\n|----------|--------|\n"
for _, row in summary.iterrows():
    table += f"| {row['category']} | {row['chunks']} |\n"
table += f"| **Total** | **{collection.count()}** |\n"

display(Markdown(table))

## 3.1 Data Overview
**Task:** Show how many files we have per folder and file type.

In [ ]:
# 3.1 data overview

files = []
for f in DATA_PATH.rglob("*"):
    if f.suffix.lower() in [".pdf", ".png", ".webp", ".jpg"]:
        files.append({"folder": f.parent.name, "type": f.suffix.lower()})

df = pd.DataFrame(files)
print(f"Files found: {len(df)}")

fig = px.bar(
    df.groupby(["folder","type"]).size().reset_index(name="count"),
    x="folder", y="count", color="type",
    title="Files per folder — KnowSeek.ai",
    barmode="group",
    color_discrete_map={".pdf": "#10B981", ".png": "#0EA5E9", ".webp": "#9199F4"}
)
fig.update_layout(
    paper_bgcolor="#0F0F0F", plot_bgcolor="#161616",
    font_color="#FFFFFF", legend_font_color="#FFFFFF",
    yaxis_range=[0, 9]
)
fig.show()

## 3.2 PDF Analysis
**Task:** Show how many pages and how much data we have per folder.

In [ ]:
# 3.2 PDF details

pdf_data = []
for f in DATA_PATH.rglob("*.pdf"):
    try:
        with pdfplumber.open(f) as pdf:
            pdf_data.append({
                "folder":  f.parent.name,
                "pages":   len(pdf.pages),
                "size_kb": round(f.stat().st_size / 1024, 1)
            })
    except Exception as e:
        print(f"Could not read file: {e}")

df_pdf = pd.DataFrame(pdf_data)

print(f"Total PDFs:   {len(df_pdf)}")
print(f"Total pages:  {df_pdf['pages'].sum()}")
print(f"Avg pages:    {round(df_pdf['pages'].mean(), 1)}")
print(f"Total size:   {round(df_pdf['size_kb'].sum() / 1024, 2)} MB")

fig = px.bar(
    df_pdf.groupby("folder").agg(pages=("pages","sum")).reset_index(),
    x="folder", y="pages", color="folder",
    title="Pages per folder — DocSeek",
    color_discrete_map={
        "01_Fasteners":    "#9199F4",
        "02_Specifikation": "#10B981",
        "03_Painting":      "#0EA5E9"
    }
)
fig.update_layout(
    paper_bgcolor="#0F0F0F", plot_bgcolor="#161616",
    font_color="#FFFFFF", legend_font_color="#FFFFFF"
)
fig.show()

## 3.3 Chunk Size Analysis
**Task:** Show how the text is split into chunks — how big are the pieces per category?

In [ ]:
# 3.3 chunk size analysis

sample = collection.get(limit=collection.count(), include=["documents", "metadatas"])
df_chunks = pd.DataFrame(sample["metadatas"])
df_chunks["chunk_length"] = [len(d) for d in sample["documents"]]

bins   = [0, 200, 300, 400, 500, 10000]
labels = ["0-200", "200-300", "300-400", "400-500", "500+"]
df_chunks["size_group"] = pd.cut(df_chunks["chunk_length"], bins=bins, labels=labels)

print(f"Total chunks:     {len(df_chunks)}")
print(f"Avg chunk length: {round(df_chunks['chunk_length'].mean())} chars")
print(f"Min:              {df_chunks['chunk_length'].min()} chars")
print(f"Max:              {df_chunks['chunk_length'].max()} chars")
print()
print(df_chunks.groupby(["size_group","category"], observed=False).size().reset_index(name="count").to_string(index=False))

df_grouped = df_chunks.groupby(["size_group","category"], observed=False).size().reset_index(name="count")

fig = px.bar(
    df_grouped, x="size_group", y="count", color="category",
    title="Chunk size distribution — by category",
    labels={"size_group": "Characters per chunk", "count": "Number of chunks"},
    barmode="group",
    color_discrete_map={
        "Corrosion":    "#10B981",
        "Painting":     "#0EA5E9",
        "Bolts+Torque": "#9199F4",
        "General":      "#FC9D57"
    }
)
fig.update_layout(
    paper_bgcolor="#0F0F0F", plot_bgcolor="#161616",
    font_color="#FFFFFF", legend_font_color="#FFFFFF"
)
fig.show()

## 3.4 Language Distribution
**Task:** Show how many documents are in German vs English.
**Note:** Only relevant for DocSeek — PartSeek uses technical drawings (no language).

In [ ]:
# 3.4 language distribution

sample = collection.get(limit=collection.count(), include=["metadatas"])
df_lang = pd.DataFrame(sample["metadatas"])

lang_count = df_lang.groupby("language", observed=False).size().reset_index(name="chunks")
print("Language distribution:")
print(lang_count.to_string(index=False))

fig = px.pie(
    lang_count, names="language", values="chunks",
    title="Language distribution — DocSeek",
    color_discrete_map={"EN": "#10B981", "DE": "#0EA5E9"}
)
fig.update_layout(
    paper_bgcolor="#0F0F0F", plot_bgcolor="#161616",
    font_color="#FFFFFF", legend_font_color="#FFFFFF"
)
fig.show()

---
# Chapter 4 — BM25 Baseline
> **Goal:** Test the keyword search (BM25) as a baseline to compare with the AI search.

BM25 searches by exact keywords — no understanding of meaning.
We measure: results + time — then log to MLFlow.

In [ ]:
# ─── Chapter 4 — Imports ──────────────────
import time
import mlflow
import pandas as pd
from pathlib import Path
from rank_bm25 import BM25Okapi
import chromadb

# ─── Global Paths — used in this chapter ──
BASE_PATH = Path.cwd().parent
DB_PATH   = str(BASE_PATH / "chroma_db")

# Check MLFlow
try:
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.search_experiments()
    print("✅ Chapter 4 imports ready")
    print("✅ MLFlow running")
except Exception:
    print("✅ Chapter 4 imports ready")
    print("⚠️  MLFlow not running — start with: mlflow ui")

## 4.1 BM25 Setup
**Task:** Load all text chunks from ChromaDB and build the BM25 index.

In [ ]:
# 4.1 Build BM25 index

client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(name="docseek")

sample    = collection.get(limit=collection.count(), include=["documents", "metadatas"])
corpus    = sample["documents"]
tokenized = [doc.lower().split() for doc in corpus]
bm25      = BM25Okapi(tokenized)

print(f"✅ BM25 index built")
print(f"   Chunks indexed: {len(corpus)}")
print(f"   Avg tokens:     {round(sum(len(t) for t in tokenized) / len(tokenized))}")

## 4.2 BM25 Search Test
**Task:** Run test queries and measure the results + time.

In [ ]:
# 4.2 Test BM25 search

test_queries = [
    "salt spray test corrosion",
    "coating standard requirements",
    "screw M16 zinc coating",
    "corrosion performance OEM",
    "cathodic e-coating specification"
]

bm25_rows = []
for query in test_queries:
    start      = time.time()
    scores     = bm25.get_scores(query.lower().split())
    best_score = round(float(scores.max()), 3)
    elapsed    = round((time.time() - start) * 1000, 2)
    bm25_rows.append({"query": query, "score": best_score, "time_ms": elapsed})
    print(f"  {query[:40]:40} score={best_score:.3f}  time={elapsed}ms")

df_bm25 = pd.DataFrame(bm25_rows)
print()
print(f"Avg score:   {round(df_bm25['score'].mean(), 3)}")
print(f"Avg time ms: {round(df_bm25['time_ms'].mean(), 2)}")

## 4.3 Log BM25 Results to MLFlow
**Task:** Save the BM25 results in MLFlow so we can compare later with RAG.

In [ ]:
# 4.3 Log results to MLFlow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("KnowSeek — BM25 vs RAG")

with mlflow.start_run(run_name="BM25 Baseline"):
    mlflow.log_param("model",     "BM25")
    mlflow.log_param("documents", len(corpus))
    mlflow.log_metric("avg_score",   round(df_bm25["score"].mean(), 3))
    mlflow.log_metric("avg_time_ms", round(df_bm25["time_ms"].mean(), 2))

print("✅ BM25 results logged to MLFlow")
print(f"   avg_score:   {round(df_bm25['score'].mean(), 3)}")
print(f"   avg_time_ms: {round(df_bm25['time_ms'].mean(), 2)}")

---
# Chapter 5 — RAG + llama3
> **Goal:** Test the AI search (RAG) for both DocSeek and PartSeek.

RAG = Retrieval Augmented Generation
- Step 1: Find the right chunks in ChromaDB
- Step 2: Send them to llama3
- Step 3: Get answer + source + confidence

In [ ]:
# ─── Chapter 5 — Imports ──────────────────
import time
import requests
import mlflow
import pandas as pd
from pathlib import Path
import sys

# ─── Global Paths — used in this chapter ──
BASE_PATH = Path.cwd().parent
DB_PATH   = str(BASE_PATH / "chroma_db")

# ─── Module Paths ─────────────────────────
sys.path.append(str(BASE_PATH / "01_backend/modules/02_docseek"))
sys.path.append(str(BASE_PATH / "01_backend/modules/01_partseek"))

import search
import answer
import search  as partseek_search
import answer  as partseek_answer

print("✅ Chapter 5 imports ready")
print("✅ DocSeek  — search + answer loaded")
print("✅ PartSeek — search + answer loaded")

## 5.1 Prompt Engineering
**Task:** Build the system prompt that tells llama3 how to behave.

In [ ]:
# 5.1 System prompt — defined in answer.py
# Shown here for reference only

SYSTEM_PROMPT = """
You are a professional engineering assistant.
You only answer based on the documents provided.
Always include the source document and page number.
If you are not sure, say so — do not guess.
If you find related or partial information — share it but mark it as:
'Based on similar content:'
"""

print("System prompt:")
print(SYSTEM_PROMPT)

## 5.2 DocSeek — RAG Search + Answer
**Task:** Run test queries through the full DocSeek RAG pipeline.

In [ ]:
# 5.2 DocSeek — Test RAG search and answer

result = answer.ask(
    "What is the salt spray test duration in hours?",
    verbose=False
)
print(f"Q: {result['question']}")
print(f"A: {result['answer'][:300]}")
print(f"Confidence: {result['signal_icon']} {result['confidence']:.3f}")
print()

print("=" * 50)
print("OEM COMPARISON")
print("=" * 50)
comparison = answer.compare_oems(
    "salt spray test duration hours closed open vehicles",
    verbose=True
)

## 5.3 PartSeek — Part Search
**Task:** Run test queries through the PartSeek search pipeline.

In [ ]:
# 5.3 PartSeek — Test part search

result = partseek_answer.find_part(
    "M8 Torx screw steel 10.9",
    verbose=True
)
print()

# With OEM filter
result_filtered = partseek_answer.find_part_with_filter(
    "flange screw",
    oem_code="OEM-V",
    verbose=True
)

## 5.4 Source Attribution
**Task:** Make sure every answer shows exactly where the information comes from.

In [ ]:
# 5.4 show source attribution

result = answer.ask(
    "What are the corrosion requirements?",
    verbose=False
)

print(f"Question: {result['question']}")
print()
print(f"Answer:   {result['answer'][:200]}...")
print()
print(f"Confidence: {result['signal_icon']} {result['confidence']:.3f}")
print()
print("Sources:")
for s in result["sources"]:
    icon = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}.get(s["signal"], "⚪")
    print(f"  {icon} Page {s['page']} — OEM: {s['oem_code']} — Score: {s['score']:.3f}")

## 5.5 Log RAG Results to MLFlow
**Task:** Save the RAG results in MLFlow so we can compare with BM25.

In [ ]:
# 5.5 Log RAG results to MLFlow

import chromadb
client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(name="docseek")

test_queries = [
    "salt spray test corrosion requirements",
    "coating standard specifications",
    "corrosion performance OEM requirements"
]

rag_rows = []
for query in test_queries:
    start  = time.time()
    result = answer.ask(query, verbose=False)
    elapsed = round((time.time() - start) * 1000, 1)
    rag_rows.append({"query": query, "score": result["confidence"], "time_ms": elapsed})

df_rag = pd.DataFrame(rag_rows)

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("KnowSeek — BM25 vs RAG")

with mlflow.start_run(run_name="RAG llama3"):
    mlflow.log_param("model",     "RAG + llama3")
    mlflow.log_param("embedding", "nomic-embed-text")
    mlflow.log_param("chunks",    collection.count())
    mlflow.log_param("n_results", 5)
    mlflow.log_metric("avg_score",   round(df_rag["score"].mean(), 3))
    mlflow.log_metric("avg_time_ms", round(df_rag["time_ms"].mean(), 1))

print("✅ RAG results logged to MLFlow")
print(f"   avg_score:   {round(df_rag['score'].mean(), 3)}")
print(f"   avg_time_ms: {round(df_rag['time_ms'].mean(), 1)}")

---
# Chapter 6 — Compare BM25 vs RAG
> **Goal:** Show clearly how BM25 and RAG differ in features and speed.

We compare:
- Features (what each model can do)
- Answer time (ms)
- Confidence score distribution

In [ ]:
# ─── Chapter 6 — Imports ──────────────────
import sys
import time
import mlflow
import pandas as pd
from pathlib import Path
import plotly.express as px
from IPython.display import display
from rank_bm25 import BM25Okapi
import chromadb

# ─── Global Paths — used in this chapter ──
BASE_PATH = Path.cwd().parent
DB_PATH   = str(BASE_PATH / "chroma_db")

# ─── Module Paths ─────────────────────────
sys.path.append(str(BASE_PATH / "01_backend/modules/02_docseek"))
import search
import answer

# ─── MLFlow ───────────────────────────────
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# ─── BM25 Index ───────────────────────────
client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(name="docseek")
sample     = collection.get(limit=collection.count(), include=["documents"])
corpus     = sample["documents"]
tokenized  = [doc.lower().split() for doc in corpus]
bm25       = BM25Okapi(tokenized)

print("✅ Chapter 6 imports ready")

## 6.1 Model Comparison — Features & Speed
**Task:** Compare BM25 and RAG on what matters for engineers.

> Note: BM25 and RAG use different scoring systems and cannot be directly compared by score.
> We compare them on features and answer time instead.

In [ ]:
# 6.1 BM25 vs RAG — same questions, fair comparison

test_queries = [
    "salt spray test corrosion requirements",
    "coating standard specifications",
    "corrosion performance OEM requirements"
]

bm25_times, rag_times = [], []

for query in test_queries:
    t1 = time.time()
    bm25.get_scores(query.lower().split())
    bm25_times.append(round((time.time() - t1) * 1000, 2))

    t2 = time.time()
    answer.ask(query, verbose=False)
    rag_times.append(round((time.time() - t2) * 1000, 2))

summary = pd.DataFrame([
    {"What we measure":  "Answer speed",
     "BM25":             f"{round(sum(bm25_times)/3, 1)}ms",
     "RAG + llama3":     f"{round(sum(rag_times)/3, 1)}ms"},
    {"What we measure":  "Finds relevant content",  "BM25": "🟡", "RAG + llama3": "🟢"},
    {"What we measure":  "Gives full answer",        "BM25": "🔴", "RAG + llama3": "🟢"},
    {"What we measure":  "Shows source + page",      "BM25": "🔴", "RAG + llama3": "🟢"},
    {"What we measure":  "Works in DE + EN",         "BM25": "🟡", "RAG + llama3": "🟢"},
    {"What we measure":  "Confidence signal 🟢🟡🔴", "BM25": "🔴", "RAG + llama3": "🟢"},
])

def color_cell(val):
    if "🟢" in str(val): return "background-color:#064E3B; color:#10B981; font-weight:bold"
    if "🟡" in str(val): return "background-color:#451A03; color:#F59E0B; font-weight:bold"
    if "🔴" in str(val): return "background-color:#450A0A; color:#EF4444; font-weight:bold"
    return "color:#E5E7EB"

display(summary.style
    .map(color_cell, subset=["BM25", "RAG + llama3"])
    .set_properties(**{"background-color": "#0F0F0F", "color": "#F3F4F6"})
    .hide(axis="index")
)

## 6.2 Answer Time — BM25 vs RAG
**Task:** How long does each model take to answer?
> BM25 is faster — RAG takes longer but gives a complete answer.

In [ ]:
# 6.2 Answer Time Comparison — from MLFlow

runs = mlflow.search_runs(experiment_names=["KnowSeek — BM25 vs RAG"])

bm25_time = runs[runs["tags.mlflow.runName"] == "BM25 Baseline"]["metrics.avg_time_ms"].values[0]
rag_time  = runs[runs["tags.mlflow.runName"] == "RAG llama3"]["metrics.avg_time_ms"].values[0]

timing = pd.DataFrame([
    {"model": "BM25 Baseline", "avg_time_ms": round(bm25_time, 1)},
    {"model": "RAG + llama3",  "avg_time_ms": round(rag_time, 1)},
])

print(f"BM25 avg time: {bm25_time:.1f}ms")
print(f"RAG  avg time: {rag_time:.1f}ms")

fig = px.bar(
    timing, x="model", y="avg_time_ms",
    title="Answer Time — BM25 vs RAG (ms)",
    color="model",
    color_discrete_map={"BM25 Baseline": "#888780", "RAG + llama3": "#10B981"},
    text="avg_time_ms",
    log_y=True
)
fig.update_layout(
    paper_bgcolor="#0F0F0F", plot_bgcolor="#161616",
    font_color="#FFFFFF", legend_font_color="#FFFFFF",
    showlegend=False
)
fig.show()

## 6.3 Confidence Score Distribution — RAG
**Task:** Show how confident the AI is for different types of questions.

🟢 > 85% — Reliable · 🟡 60–85% — Check source · 🔴 < 60% — Verify manually

In [ ]:
# 6.3 Confidence Score Distribution — real data from ChromaDB

test_queries = [
    "corrosion performance standard requirements",
    "salt spray test duration hours",
    "which OEM has the worst case salt spray test requirement",
    "flange screw class 10.9 M10 30mm length",
    "unknown part XYZ random query 999"
]

conf_rows = []
for query in test_queries:
    results = search.search(query, n_results=1, verbose=False)
    if results:
        r = results[0]
        conf_rows.append({
            "query":      query[:35] + "...",
            "confidence": r["score"],
            "signal":     r["signal"]
        })

df_conf = pd.DataFrame(conf_rows)

fig = px.bar(
    df_conf, x="query", y="confidence",
    color="signal",
    title="Confidence Score per Query — RAG",
    color_discrete_map={"GREEN": "#10B981", "YELLOW": "#F59E0B", "RED": "#EF4444"},
    text="confidence"
)
fig.update_xaxes(tickangle=20)
fig.add_hline(y=0.85, line_dash="dash", line_color="#10B981", annotation_text="🟢 85%")
fig.add_hline(y=0.60, line_dash="dash", line_color="#F59E0B", annotation_text="🟡 60%")
fig.update_layout(
    paper_bgcolor="#0F0F0F", plot_bgcolor="#161616",
    font_color="#FFFFFF", legend_font_color="#FFFFFF",
    yaxis_range=[0, 1]
)
fig.show()

---
*KnowSeek.Ai — Version rev05_003 — Branch main_sia05*

*All data stays local. No cloud. No internet required.*